# Run and Inspect Containers

This notebook introduces running and inspecting local Docker containers.

## Prepare one image

Run these commands only against the authorized local base-station Docker daemon; do not point the Docker client at a Duckiedrone, another person's computer, or an unfamiliar context.

An image must be available locally before Docker can create a container from it. Pull the small, explicitly tagged `alpine:3.21` image:

```bash
docker pull alpine:3.21
docker image inspect --format '{{.Os}}/{{.Architecture}}' alpine:3.21
docker image history alpine:3.21
```

`docker pull` downloads image layers when they are not already in the local cache. `docker image inspect` reports the platform of the local image variant. `docker image history` lists the image layers. It is normal for a multi-platform image to resolve to the platform of the local Docker daemon.

Pulling an image changes only the local image cache. Do not remove images merely because you did not create them, and do not use broad cleanup commands. The explicit `alpine:3.21` tag makes the example repeatable without relying on `latest`.

## Run a temporary command

`docker run` creates a new container and starts its main process. Run one short command in a temporary container:

```bash
docker run --rm --name lx-docker-hello alpine:3.21 sh -c 'printf "%s\n" "Hello from a temporary container"; cat /etc/os-release'
```

`--name` gives the container a predictable name. `--rm` removes this practice container after its command exits. The `sh -c` argument asks the container's shell to run both commands. The output comes from inside the container, but it appears in the base-station terminal.

Run the command a second time if you want to confirm that each `docker run` creates a fresh container. Because `--rm` is present, no stopped `lx-docker-hello` container should remain.

## Keep a container running

Some containers run a service for longer than one command. Start a clearly named practice container in the background:

```bash
docker run --detach --name lx-docker-sleeper alpine:3.21 sh -c 'printf "%s\n" "The practice container is running"; exec sleep 300'
docker container ls --filter 'name=lx-docker-sleeper'
docker logs lx-docker-sleeper
```

`--detach` starts the container in the background and prints its container identifier (ID). `docker container ls` shows the running container, and `docker logs` shows the text written by its main process. The `exec` command replaces the temporary shell with `sleep 300`, so the container remains running for up to five minutes unless you stop it first.

## Observe a running container

Use read-only Docker commands to gather evidence before deciding why a local container is slow, stopped, or unexpectedly large:

```bash
docker stats --no-stream lx-docker-sleeper
docker top lx-docker-sleeper
docker container inspect --format 'status={{.State.Status}} started={{.State.StartedAt}}' lx-docker-sleeper
docker system df
```

`docker stats` normally refreshes continuously. `--no-stream` prints one snapshot and exits, showing CPU use, memory use and limit, network and block input/output (I/O), and the process count for the named container. The sleeper should use very little CPU. To watch a container over time, omit `--no-stream` and press `Ctrl-C` to stop the display; this does not stop the container.

`docker top` lists the processes running inside the named container, which complements host-level `ps` or `htop`. `docker container inspect --format ...` selects two fields from Docker's structured container data instead of printing the full JavaScript Object Notation (JSON) document. `docker system df` reports the local daemon's aggregate image, container, volume, and build-cache disk use; it reports information only and does not remove anything.

All four commands inspect the Docker daemon selected by the current context. In this LX, leave that context local and use only `lx-docker-sleeper`. On another Docker host, first confirm that you own it or have explicit permission to inspect it. Resource values are evidence, not a diagnosis: compare a named container's measurements over time, then inspect its logs and documented configuration before changing anything.

## Open an additional shell

Use `docker exec` to start an extra shell in the already running container:

```bash
docker exec --interactive --tty lx-docker-sleeper sh
```

Inside that shell, try these non-destructive commands:

```bash
pwd
whoami
cat /etc/os-release
exit
```

`docker exec` starts a new process in the named running container. `--interactive` keeps standard input open and `--tty` gives the shell terminal behavior. `exit` closes only the extra shell; the main `sleep` process continues to keep the container running.

`docker attach` connects directly to a container's main process, which can make its input and lifecycle harder to understand for a beginner. Use `docker exec` for this LX.

## Stop the practice container

When you are finished, stop exactly the container created above:

```bash
docker stop lx-docker-sleeper
docker container ls --all --filter 'name=lx-docker-sleeper'
docker container rm lx-docker-sleeper
```

`docker stop` requests a clean shutdown of the main process. The filtered list should now show a stopped `lx-docker-sleeper` container. `docker container rm` removes that exact stopped practice container. Unlike the earlier one-command example, this container does not use `--rm` so you can observe its stopped state first.

If a command reports that the name is already in use, inspect only that exact name with `docker container ls --all --filter 'name=lx-docker-sleeper'`. If it is a previous container you created for this LX, complete the stop-and-remove steps above; otherwise, do not remove an unfamiliar container to make room.

## Container lifecycle summary

```text
image available locally
  |
  v
docker run creates a container
  |
  v
main process runs
  |
  v
process exits or docker stop stops it
  |
  v
docker container rm removes an explicitly named container
```

The container exits when its main process exits. An interactive shell, a web server, and the `sleep` command are all possible main processes. A stopped container can remain for later inspection unless `--rm` was used. The earlier `lx-docker-hello` example uses `--rm`; the longer `lx-docker-sleeper` example shows explicit removal instead.

## Further reading

Docker's official [container introduction](https://docs.docker.com/get-started/docker-concepts/the-basics/what-is-a-container/) and [`docker container run` reference](https://docs.docker.com/reference/cli/docker/container/run/) explain the lifecycle and options used here. The [`docker stats` reference](https://docs.docker.com/reference/cli/docker/container/stats/) explains its columns and formatting options. The [`docker container exec` reference](https://docs.docker.com/reference/cli/docker/container/exec/) describes additional shell and command options.

## Checkpoint

Run the self-check in the next cell. Write or select a response before revealing the answer.


In [ ]:
import sys
from pathlib import Path

working_directory = Path.cwd()
parent_directory = working_directory.parent
if (parent_directory / "packages").is_dir():
    parent_directory_path = str(parent_directory)
    sys.path.insert(0, parent_directory_path)

from packages.checkpoint_self_check import display_checkpoint_self_checks

display_checkpoint_self_checks()
